# Exercise 2.4: Transforming Data & Creating New Features (Angola IEA)

This notebook turns the cleaned file into an analysis ready dataset, and ends by
producing Angola's headline labour market statistic two different ways.

You will practice:
- Banding a continuous variable with `pd.cut()`
- Decoding numeric codes with `.map()` and a dictionary
- Building a three way category with `np.select()`
- Binary flags with `np.where()` and updates with `.loc[]`
- Chained, dependent columns with `assign()` and lambdas
- Custom row logic with `apply()`
- Weighting a statistic, and seeing why the definition matters more than the code

> **Pipeline:** run Exercise 2.3 first. Writes to `20_processed/`.

### Path Setup (run first)

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'
clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

STR_COLS = {
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
}
df = pd.read_csv(clean_path, dtype=STR_COLS)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

---

## Task 1: Age bands with `pd.cut()`

The labour statistics that follow all rest on the working age population, which
Angola defines as 15 and over. Band the ages accordingly.

Note `right=False`, which makes each interval closed on the left: a 15 year old
belongs to Youth, not Child.

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 15, 25, 65, 120],
    labels=  # your code here: Child, Youth, Adult, Elderly
    right=  # your code here
)
df['age_group'].value_counts(dropna=False)

**Questions:**

- How many people fall into each age group, and do the counts sum to the full
  sample?
- What would change about the working age population if you used the default
  `right=True` instead of `right=False`?
- What share of the sample is Children, and why is that the single most
  important fact about Angola's labour market?

---

## Task 2: Decode the numeric codes with `.map()`

The codebook read in 2.1 gives the code to label mapping. `.map()` applies a
dictionary element by element and returns `NaN` for anything not in it, which
doubles as a check.

In [ ]:
PROVINCE_MAP = {
    '10': 'Cabinda', '11': 'Zaire', '12': 'Uíge', '13': 'Bengo', '14': 'Luanda',
    '15': 'Cuanza-Norte', '16': 'Cuanza-Sul', '17': 'Malanje', '18': 'Lunda-Norte',
    '19': 'Lunda-Sul', '20': 'Moxico', '21': 'Bié', '22': 'Huambo', '23': 'Benguela',
    '24': 'Namibe', '25': 'Huila', '26': 'Cunene', '27': 'Cubango',
    '28': 'Icolo e Bengo', '29': 'Moxico Leste', '30': 'Cuando',
}
SEX_MAP = {1: 'Masculino', 2: 'Feminino'}
AREA_MAP = {1: 'Urbana', 2: 'Rural'}
EDUCATION_MAP = {
    1: 'Primário', 2: 'I Ciclo Secundário', 3: 'II Ciclo Secundário',
    4: 'Bacharelato', 5: 'Licenciatura', 6: 'Mestrado', 7: 'Doutoramento',
    9: 'Nenhum nível',
}

df['province_name'] = df['province_code'].  # your code here: map(PROVINCE_MAP)
df['sex_label'] = df['sex'].map(SEX_MAP)
df['area_label'] = df['area_type'].map(AREA_MAP)
df['education_label'] = df['education_level'].map(EDUCATION_MAP)

print('Unmapped provinces:', df['province_name'].isna().sum())
print(df['area_label'].value_counts(dropna=False))
print()
print(df['education_label'].value_counts(dropna=False))

**Questions:**

- How many unmapped provinces are there? Why does building the dictionary from
  the file's own codebook rather than from memory matter?
- How many people are urban versus rural?
- Why is `education_label` `NaN` for the majority of rows, and why can't
  `.map()` alone tell you whether a `NaN` was genuinely missing or simply an
  unmapped code?
- Why is it wrong to treat the education codes as ordinal? What happens to
  province coverage if the mapping is copied from an older publication?

---

## Task 3: Labour force status with `np.select()`

`np.select()` takes conditions in order and applies the first match. This is the
heart of the notebook, and the definitions matter more than the syntax.

**Employed:** worked for pay, or worked on own account, or has a job they were
absent from. **Unemployed (strict ILO):** not employed, actively looked for work,
and available to start. Everyone else of working age is outside the labour force.

Note what this rule leaves out: `worked_family_business`, the flag for
contributing family workers, people who work unpaid in a household member's
farm or business. ICLS-19, the international labour statistics standard,
counts them as employed when the unit is a market one. Excluding them here is
not something the data forces on you: it is a definitional choice, made
silently unless you say so. Task 4 comes back to this and shows what the
number does when the choice is made differently.

In [ ]:
working_age = df['age'] >= 15
employed = (
    (df['worked_for_pay'] == 1)
    | (df['worked_own_account'] == 1)
    | (df['absent_from_job'] == 1)
)
seeking = (df['sought_work'] == 1) | (df['sought_business'] == 1)
available = (df['available_now'] == 1)   # your code here: also allow available_2wk

strict_conditions = [working_age & employed, working_age & seeking & available]
df['lf_status_strict'] = np.select(  # your code here: conditions, choices, default )

df['lf_status_strict'].value_counts()

In [ ]:
# The relaxed definition also counts people who want work but have stopped
# looking: the discouraged, who a strict measure treats as economically inactive.
relaxed_conditions = [
    working_age & employed,
    working_age & (  # your code here: seeking & available, or wants_work )
]
df['lf_status_relaxed'] = np.select(
    relaxed_conditions, ['Employed', 'Unemployed'], default='Outside labour force')

df['lf_status_relaxed'].value_counts()

**Questions:**

- What are the strict definition counts for Employed, Unemployed and Outside
  labour force?
- What changes under the relaxed definition, and roughly how many people move
  from outside the labour force into unemployed?
- Why are `available_now` and `available_2wk` combined with `|` instead of
  using `available_2wk` alone?
- Why does the order of conditions passed to `np.select()` matter here?

---

## Task 4: Weight the result

Each person in the sample stands for many people in Angola, and `weight_ind`
records how many. An unweighted rate describes the sample; a weighted rate
describes the country. Published statistics are always weighted.

In [ ]:
def unemployment_rate(status, weights):
    """Unemployment as a percentage of the labour force."""
    unemployed = status == 'Unemployed'
    labour_force = status.isin(['Employed', 'Unemployed'])
    # your code here: weighted unemployed over weighted labour force
    return


weight = df['weight_ind']
ones = pd.Series(1, index=df.index)

for name in ['lf_status_strict', 'lf_status_relaxed']:
    print(f'{name:20s} weighted: {unemployment_rate(df[name], weight):5.1f}%'
          f'   unweighted: {unemployment_rate(df[name], ones):5.1f}%')

In [ ]:
in_labour_force = df['lf_status_strict'].isin(['Employed', 'Unemployed'])
participation = weight[in_labour_force].sum() / weight[working_age].sum() * 100
print(f'Labour force participation rate: {participation:.1f}%')
print(f'Weighted working age population: {weight[working_age].sum():,.0f}')

In [ ]:
# Contributing family workers: employed under ICLS-19, excluded by the rule
# in Task 3. This is additive, it does not change lf_status_strict itself.
employed_with_family = employed   # your code here: also count worked_family_business == 1
family_status = np.select(
    [working_age & employed_with_family, working_age & seeking & available],
    ['Employed', 'Unemployed'], default='Outside labour force')
family_status = pd.Series(family_status, index=df.index)

print('Working-age contributing family workers not otherwise employed:',
      (working_age & ~employed & (df['worked_family_business'] == 1)).sum())
print('Strict rate excluding them: %.1f%%' % unemployment_rate(df['lf_status_strict'], weight))
print('Strict rate including them: %.1f%%' % unemployment_rate(family_status, weight))

**Questions:**

- What are the weighted strict and relaxed unemployment rates, and how large is
  the gap between them?
- What does INE Angola publish as its headline figure, and which of the two
  measures does it correspond to?
- How much do the weights move the strict unemployment estimate, and why is
  that not a reason to skip weighting elsewhere?
- Which number belongs in a press release, and why must the definition used be
  stated explicitly?
- If contributing family workers are counted as employed under ICLS-19, how many
  working-age people does that add, and where does strict unemployment land?

---

## Task 5: Binary flags with `np.where()` and `.loc[]`

`np.where()` is a vectorised if/else. `.loc[]` updates values that match a
condition, which is how you add a third state afterwards.

In [ ]:
df['full_time'] = np.where(df['hours_usual'] >= 35, 'Full time', 'Part time')

# np.where has no idea what a missing value means: it lands in the else branch.
# Make the unknown explicit instead of letting it masquerade as part time.
df.loc[  # your code here: rows where hours_usual is missing , 'full_time'] = 'Unknown'

df['full_time'].value_counts()

**Questions:**

- What are the full time, part time and unknown counts?
- What would happen to the part time count if the `.loc[]` line for missing
  hours were left out?
- Who are the people in the unknown group, mostly?

---

## Task 6: Dependent columns with `assign()`

`assign()` returns a new DataFrame, so it chains. A lambda inside it sees the
frame **as it is being built**, which is how the second column below can use the
first one created in the same call.

In [ ]:
df = df.assign(
    job_tenure_years=lambda x:   # your code here
    tenure_band=lambda x: np.select(
        [x['job_tenure_years'] < 1,
         x['job_tenure_years'] < 5,
         x['job_tenure_years'] >= 5],
        ['Under 1 year', '1 to 4 years', '5 years or more'],
        default='Unknown',
    ),
)
df['tenure_band'].value_counts()

**Questions:**

- How many people fall into each tenure band?
- Why must `tenure_band` reference `lambda x: x['job_tenure_years']` rather
  than `df['job_tenure_years']`?
- Why does Unknown dominate the tenure band counts?

---

## Task 7: Household size, without `groupby`

`hh_size_reported` was dropped in 2.3 because it was empty. Rebuild it from the
roster: count how many rows share each `household_id`, then map that count back
onto every person.

`value_counts()` comes from 2.1 and `.map()` from 2.4, so no new tool is needed.

In [ ]:
df['hh_size'] = df['household_id'].  # your code here: map with value_counts()

print('Mean household size per person:   ', round(df['hh_size'].mean(), 2))
print('Mean household size per household:',
      round(df.drop_duplicates('household_id')['hh_size'].mean(), 2))
df['hh_size'].describe()

**Questions:**

- What is the mean household size per person, and what is it per household?
  Why do they differ?
- Why does averaging `hh_size` over rows over-weight large households?
- Why is publishing the per person figure as "average household size" a
  mistake?

---

## Task 8: Custom logic with `apply()`

When built in operations cannot express the rule, `apply()` runs your own
function. It processes rows one at a time and is much slower than a vectorised
operation, so reach for it last, not first.

In [ ]:
def hours_per_day(row):
    """Usual weekly hours spread over 7 days, or NaN when the input is unusable."""
    hours = row['hours_usual']
    # your code here: guard NaN and non positive, then round(hours / 7, 2)


df['hours_per_day'] = df.apply(hours_per_day, axis=1)
print('Mean hours per day:', round(df['hours_per_day'].mean(), 2))
df[['household_id', 'hours_usual', 'hours_per_day']].dropna().head()

**Questions:**

- What is the mean hours per day across people who report any hours?
- What does `axis=1` do in `df.apply(hours_per_day, axis=1)`, and what would
  happen without it?
- Why would this particular calculation be faster written as a vectorised
  expression instead of `apply()`?

---

## Task 9: Save the feature table

Cleaned data lives in `10_cleaned/`. Derived, analysis ready tables go in
`20_processed/`.

In [ ]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)
out_path = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')

df.  # your code here: to_csv with index=False
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype=STR_COLS)
print('Reloaded:', check.shape)
print('Columns added since the cleaned file:', check.shape[1] - 27)
check[['household_id', 'age_group', 'province_name',
       'lf_status_strict', 'lf_status_relaxed', 'hh_size']].head()

**Questions:**

- What is the final shape of the features file, and how many columns were
  added since the cleaned file?
- Which operations in this notebook are vectorised, and which one is not?
- Does the row count change between 2.3 and 2.4? What is the point of that?